# Gait-ViViT: A Video Processing Model for Parkinson's Disease Detection

This Google Colab-hosted Jupyter Notebook contains code and markdown explanations for the internal internship that I completed at VisionLab in Sapienza University of Rome, under the supervision of Professor Marini and Dr. Diko, in the field of medical video processing.

This project focuses on developing a video classification model that, given a video sequence of a person walking, aims to detect whether that person is afflicted with Parkinson's disease or not.

## 0 - General Requirements

In [ ]:
# Dataset Setup
import os
import json
import cv2
import numpy as np
import subprocess
from pathlib import Path
from glob import glob
import torch
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
try:
  import imageio
except ImportError:
  # Use imageio with ffmpeg for video format compatibility.
  !pip install imageio imageio-ffmpeg
  import imageio
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as v2
from torchvision import tv_tensors
import matplotlib.pyplot as plt
from transformers import VivitModel, VivitForVideoClassification
import torch.nn as nn
import torch.nn.functional as F
import math
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from torch.amp import GradScaler, autocast
import matplotlib.pyplot as plt
import seaborn as sns
import torch.optim as optim
from torch.optim import lr_scheduler
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from transformers import get_cosine_schedule_with_warmup

## 1 - Dataset Setup

The datasets used for this project are [Connie et al.'s Kaggle MMU Visual-Based Parkinson's Disease Dataset](https://www.kaggle.com/datasets/teeconnie/mmu-visual-based-parkinsons-disease-dataset), which is hosted on Kaggle, and an internal dataset from Sapienza University of Rome.

### Kaggle MMU Visual-Based Parkinson's Disease Dataset

This dataset contains data extracted from **292 videos** of 167 subjects, of whom **93 are healthy** and **74 have Parkinson's disease**, walking.

The dataset is organized in **four folders**.
| Folder | Files | Description |
| :---: | :---: | :---: |
| `NORMAL` | 150 | Healthy subjects |
| `MILD` | 29 | Subjects with mild symptoms of Parkinson's disease |
| `MODERATE` | 61 | Subjects with moderate symptoms of Parkinson's disease |
| `SEVERE` | 62 | Subjects with severe symptoms of Parkinson's disease |

In [ ]:
!pip install --upgrade kaggle

In [ ]:
from google.colab import userdata

# Use the Kaggle Access Token that was saved using Google Colab Secrets.
os.environ["KAGGLE_TOKEN"] = userdata.get("KAGGLE_TOKEN")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Download and decompress the Kaggle MMU Dataset.
!mkdir -p /content/drive/MyDrive/bachelor_thesis/kaggle_json
!kaggle datasets download -d teeconnie/mmu-visual-based-parkinsons-disease-dataset -p /content/drive/MyDrive/bachelor_thesis/kaggle_json
!unzip -q -o /content/drive/MyDrive/bachelor_thesis/kaggle_json/mmu-visual-based-parkinsons-disease-dataset.zip -d /content/drive/MyDrive/bachelor_thesis/kaggle_json
!rm -f /content/drive/MyDrive/bachelor_thesis/kaggle_json/mmu-visual-based-parkinsons-disease-dataset.zip

print("Kaggle MMU Dataset successfully downloaded.")

Note that, instead the original videos, this dataset consists of just **JSON files** containing the **information extracted from each video**, likely due to privacy compliance reasons.

This information has been extracted using AlphaPose's point extraction method, which uses a heatmap for each keypoint and **extracts the keypoint with highest probability**.

For this reason, information is presented as a sequence of keypoints containing **spatial coordinates** and their **confidence**, which denotes the probability of the corresponding keypoint.

Therefore, the first step is to use the information contained in the JSON files to create **synthetic MP4 videos** containing the subject's **skeleton** during the video.

Each video contains the rendered skeleton, which is created using [AlphaPose's Halpe Full-Boy Human Keypoints](https://github.com/Fang-Haoshu/Halpe-FullBody), on a black background.

Given the structure of the JSON files in the dataset, this rendering function uses the `OpenCV` library to create the synthetic videos.

In [ ]:
# Halpe-Body Human Keypoints: https://github.com/Fang-Haoshu/Halpe-FullBody
HALPE_SKELETON = [
    (0, 1), (0, 2), (1, 3), (2, 4),               # Head and face
    (5, 18), (6, 18),                             # Shoulders and neck
    (5, 7), (7, 9),                               # Left arm
    (6, 8), (8, 10),                              # Right arm
    (18, 19),                                     # Neck and pelvis
    (11, 19), (12, 19),                           # Pelvis and hips
    (11, 13), (13, 15),                           # Left leg
    (12, 14), (14, 16),                           # Right leg
    (15, 24), (15, 20), (20, 22),                 # Left foot
    (16, 25), (16, 21), (21, 23)                  # Right foot
]

def json_to_video_kaggle(json_path, output_path, width=1280, height=720, fps=30):
  # Create the output directory.
  output_dir = os.path.dirname(output_path)
  if output_dir:
    os.makedirs(output_dir, exist_ok=True)

  # Load the JSON keypoint file.
  with open(json_path, "r") as f:
    frames_data = json.load(f)

  # Create a temporary file for saving the raw video.
  temp_output_path = output_path.replace(".mp4", "_temp.mp4")

  # Initialize OpenCV's VideoWriter to create .mp4 videos.
  fourcc = cv2.VideoWriter_fourcc(*"mp4v")
  out = cv2.VideoWriter(temp_output_path, fourcc, fps, (width, height))

  if not out.isOpened():
    raise RuntimeError(f"Error during initialization while processing {temp_output_path}.")

  for frame_info in frames_data:
    # Create a black background for each frame.
    img = np.zeros((height, width, 3), dtype=np.uint8)

    # Extract keypoints.
    people = frame_info.get("people", [frame_info])
    for person in people:
      keypoints = person.get("keypoints", [])
      points = []

      '''
      JSON data contain spatial coordinates and confidence scores.
      In fact, the model samples keypoints according to a heatmap, choosing the point with highest probability.
      The probability of the sampled point represents the model's confidence during extraction.
      '''

      for i in range(0, len(keypoints), 3):
        x, y, conf = keypoints[i], keypoints[i+1], keypoints[i+2]

        # Eliminate low-confidence points for denoising.
        if conf > 0.2:
          pt = (int(x), int(y))
          cv2.circle(img, pt, radius=4, color=(0, 255, 0), thickness=-1)
          points.append(pt)
        else:
          points.append(None)

        # Draw the bones according to the keypoint tuples.
        for connection in HALPE_SKELETON:
          idx1, idx2 = connection

          # Check that the two indices do not fall out of range.
          if idx1 < len(points) and idx2 < len(points):
            pt1 = points[idx1]
            pt2 = points[idx2]

            # Draw the bone if and only if both points have been detected.
            if pt1 is not None and pt2 is not None:
              cv2.line(img, pt1, pt2, color=(255, 0, 0), thickness=2) # Remember that OpenCV uses the BGR scheme.

    out.write(img)

  out.release()

  # Convert the raw video to the correct format.
  cmd = f"ffmpeg -y -i {temp_output_path} -c:v libx264 -pix_fmt yuv420p {output_path}"
  subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

  if os.path.exists(temp_output_path):
    os.remove(temp_output_path)

  print(f"{json_path} converted to synthetic video: {output_path}.")

In [ ]:
# Convert all JSON files in the Kaggle MMU Dataset into synthetic MP4 videos.
kaggle_json_dir = Path("/content/drive/MyDrive/bachelor_thesis/kaggle_json")
kaggle_mp4_dir = Path("/content/drive/MyDrive/bachelor_thesis/kaggle_videos")
kaggle_mp4_dir.mkdir(parents=True, exist_ok=True)

for el in kaggle_json_dir.rglob("*.json"):
  relative_json_path = el.relative_to(kaggle_json_dir)
  relative_mp4_path = relative_json_path.with_suffix(".mp4") # Change suffix from .json to .mp4.
  drive_mp4_path = kaggle_mp4_dir / relative_mp4_path
  drive_mp4_path.parent.mkdir(parents=True, exist_ok=True)

  # Since the procedure may interrupt, skip any file that has already been processed.
  if drive_mp4_path.exists():
    continue

  print(f"Converting {el} into a synthetic video.")
  json_to_video_kaggle(str(el), str(drive_mp4_path))

print("Kaggle MMU Dataset successfully rendered.")

### Internal GAIT Dataset

This dataset contains **2448 videos**, incuding depth, infrared, and RGB versions, of various **healthy subjects** walking or moving up/down the stairs.

Since the videos come in **AVI** format, the first step is to **extract spatial coordinates and confidence scores** using the AlphaPose Halpe model and using these keypoints to create **synthetic MP4 videos**.

Note that, since rendering the entire dataset can be problematic or unnecessary, only a portion of this dataset will actually be used.

Since the AlphaPose model has issues with processing videos in AVI format, the first step requires **converting the videos to MP4 format**.

In [ ]:
gait_avi = Path("/content/drive/MyDrive/bachelor_thesis/GAIT/dataset_blurred")
gait_mp4 = Path("/content/drive/MyDrive/bachelor_thesis/internal_mp4/dataset_blurred")
gait_json = Path("/content/drive/MyDrive/bachelor_thesis/internal_json/dataset_blurred")

gait_mp4.mkdir(parents=True, exist_ok=True)
gait_json.mkdir(parents=True, exist_ok=True)

# Start converting videos from .avi to .mp4 for compatibility with OpenCV modules used by AlphaPose.
avi_videos = sorted(list(gait_avi.rglob("*.avi")))
print(f"{len(avi_videos)} videos found to convert.")

for avi_path in avi_videos:
  # Create the target path for the current video.
  relative_path = avi_path.relative_to(gait_avi)
  mp4_path = gait_mp4 / relative_path.with_suffix(".mp4")
  mp4_path.parent.mkdir(parents=True, exist_ok=True)

  # Since the procedure may interrupt, skip any videos that have been previously converted.
  if not os.path.exists(mp4_path):
    print(f"Converting {relative_path}.")
    cmd = f"ffmpeg -y -i '{str(avi_path)}' -vcodec libx264 -pix_fmt yuv420p '{str(mp4_path)}' -loglevel error"
    subprocess.run(cmd, shell=True)

print("GAIT Dataset converted to .mp4 format.")

Since the rendering procedure tends to be computationally heavy, make sure to set *Runtime ▶ Change runtime type ▶ T4 GPU* before executing this section.

In [ ]:
print("A GPU is Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
# Install the AlphaPose Halpe model.
!pip install ninja yacs cython matplotlib tqdm opencv-python tensorboardX --quiet

%cd /content
if not os.path.exists("/content/AlphaPose"):
  !git clone https://github.com/MVIG-SJTU/AlphaPose.git

%cd /content/AlphaPose
!python setup.py build develop --quiet

# For simplicity, download the model only if it was not pre-downloaded.
!mkdir -p pretrained_models
drive_model_path = "/content/drive/MyDrive/bachelor_thesis/halpe26_fast_res50_256x192.pth"
if os.path.exists(drive_model_path):
  print("Copying the model from Google Drive.")
  !cp {drive_model_path} pretrained_models/
else:
  print("Downloading the model.")
  !gdown --id 1S-ROA28de-1zvLv-hVfPFJ5tFBYOSITb -O pretrained_models/halpe26_fast_res50_256x192.pth
  os.makedirs("/content/drive/MyDrive/bachelor_thesis", exist_ok=True)
  !cp pretrained_models/halpe26_fast_res50_256x192.pth {drive_model_path}

print("AlphaPose model ready for use.")

Remember to install the `cython_bbox` package and to fetch the `YOLO` weights used by the AlphaPose model.

In [ ]:
!pip install cython_bbox

In [ ]:
!mkdir -p /content/AlphaPose/detector/yolo/data
!wget -O /content/AlphaPose/detector/yolo/data/yolov3-spp.weights https://pjreddie.com/media/files/yolov3-spp.weights

In [ ]:
def process_single_video(mp4_path, is_first=False):
  # GPU-parallelized multithreaded video processing function.
  relative_mp4_path = mp4_path.relative_to(gait_mp4_root)
  target_json_file = gait_json_root / relative_mp4_path.parent / f"{mp4_path.stem}.json"

  # Since the procedure may interrupt, skip any video that has been previously processed.
  if target_json_file.exists():
    return

  # Use id(mp4_path) to avoid ambiguity when multiple threads are running in parallel.
  unique_prefix = id(mp4_path)
  local_video_path = local_temp_dir / f"{unique_prefix}_{mp4_path.name}"
  local_out_dir = local_temp_dir / f"{unique_prefix}_{mp4_path.stem}"
  local_out_dir.mkdir(parents=True, exist_ok=True)

  print(f"\n[START] Starting: {relative_mp4_path}", flush=True)

  try:
    # Work on a local copy.
    shutil.copy(str(mp4_path), str(local_video_path))
    cmd = [
          "python",
          "scripts/demo_inference.py",
          "--cfg", "configs/halpe_26/resnet/256x192_res50_lr1e-3_1x.yaml",
          "--checkpoint", checkpoint_path,
          "--video", str(local_video_path),
          "--outdir", str(local_out_dir),
          "--format", "cmu"
    ]

    # Since the first video needs to load the AlphaPose model, it gets a larger timeout.
    timeout = 360 if is_first else 180

    # Run the subprocess.
    subprocess.run(cmd, cwd="/content/AlphaPose", timeout=timeout, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

  except subprocess.TimeoutExpired:
    print(f"\n[TIMEOUT] Unlocking processes for {relative_mp4_path}", flush=True)

  except Exception as e:
    print(f"\n[ERROR] Unexpected error on {relative_mp4_path}: {e}", flush=True)

  extracted_file = local_out_dir / "alphapose-results.json"

  # Save the result on Google Drive and clean the local space on Google Colab.
  if extracted_file.exists():
    target_json_file.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(extracted_file), str(target_json_file))
    print(f"\n[SUCCESS] {relative_mp4_path}.", flush=True)
  else:
    print(f"\n[FAIL] {relative_mp4_path}.", flush=True)

  shutil.rmtree(local_out_dir, ignore_errors=True)
  if local_video_path.exists():
    local_video_path.unlink()

In [ ]:
gait_mp4_root = Path("/content/drive/MyDrive/bachelor_thesis/internal_mp4/dataset_blurred")
gait_json_root = Path("/content/drive/MyDrive/bachelor_thesis/internal_json/dataset_blurred")

mp4_videos = sorted(list(gait_mp4_root.rglob("*.mp4")))
print(f"Videos found to process: {len(mp4_videos)}")

checkpoint_path = "/content/drive/MyDrive/bachelor_thesis/halpe26_fast_res50_256x192.pth"

# Since the procedure may interrupt, skip any video that has been previously processed.
videos_to_process = []
for mp4_path in mp4_videos:
  relative_mp4_path = mp4_path.relative_to(gait_mp4_root)
  target_json_file = gait_json_root / relative_mp4_path.parent / f"{mp4_path.stem}.json"
  if not target_json_file.exists():
    videos_to_process.append(mp4_path)

print(f"{len(videos_to_process)} videos left to process.")

# Create a local folder to make the workflow less problematic.
local_temp_dir = Path("/content/temp_processing")
local_temp_dir.mkdir(parents=True, exist_ok=True)

if videos_to_process:
  # The first video is processed separately to avoid race conditions when loading the AlphaPose model.
  print("Processing the first video and initializing the AlphaPose model.")
  first_video = videos_to_process[0]
  process_single_video(first_video, is_first=True)

  # The remaining videos are processed in parallel.
  remaining_videos = videos_to_process[1:]

  # Set the number of threads based on the chosen GPU.
  # The L4 GPU (24 GB) can support 2-3 threads, whereas the A100 GPU (40 GB) can support 4-5 threads.
  MAX_WORKERS = 2

  if remaining_videos:
    print(f"Processing the remaining {len(remaining_videos)} using {MAX_WORKERS} threads.")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
      # Submit all tasks and gradually wait for completion.
      futures = [executor.submit(process_single_video, video, False) for video in remaining_videos]
      for future in as_completed(futures):
        try:
          future.result()
        except Exception as e:
          print(f"Error caused by thread: {e}", flush=True)

print("Keypoint extraction successfully completed.")

After extracting the keypoints from each video, it is possible to **render the JSON files in synthetic videos**, slightly adapting the method used for the Kaggle MMU Visual-Based Parkinson's Disease Dataset to make it compatible with how AlphaPose extracts and stores the keypoints.

Due to the format of the JSON files, this rendering function uses both the `OpenCV` and `Imageio` libraries for better flexibility.

In [ ]:
# Halpe-Body Human Keypoints: https://github.com/Fang-Haoshu/Halpe-FullBody
HALPE_SKELETON = [
    (0, 1), (0, 2), (1, 3), (2, 4),               # Head and face
    (5, 18), (6, 18),                             # Shoulders and neck
    (5, 7), (7, 9),                               # Left arm
    (6, 8), (8, 10),                              # Right arm
    (18, 19),                                     # Neck and pelvis
    (11, 19), (12, 19),                           # Pelvis and hips
    (11, 13), (13, 15),                           # Left leg
    (12, 14), (14, 16),                           # Right leg
    (15, 24), (15, 20), (20, 22),                 # Left foot
    (16, 25), (16, 21), (21, 23)                  # Right foot
]

# Adapt the width and height parameters to the original video resolution.
def json_to_video_v2(json_path, output_path, width=848, height=480, fps=30):
  # Create the output directory.
  output_dir = os.path.dirname(output_path)
  if output_dir:
    os.makedirs(output_dir, exist_ok=True)

  # Load JSON data.
  with open(json_path, 'r') as f:
    frames_dict = json.load(f)

  # Since JSON extraction is done is parallel, sort the frames first.
  sorted_frame_keys = sorted(frames_dict.keys(), key=lambda x: int(x.split('.')[0]))

  # Initialize Imageio's writer to create compatible videos.
  writer = imageio.get_writer(output_path, fps=fps, codec='libx264', pixelformat='yuv420p')

  # Iterate through frames.
  for frame_key in sorted_frame_keys:
    frame_info = frames_dict[frame_key]

    img = np.zeros((height, width, 3), dtype=np.uint8)

    # Extract data from the bodies/joints structure.
    bodies = frame_info.get('bodies', [])

    for body in bodies:
      keypoints = body.get('joints', [])
      points = []

      for i in range(0, len(keypoints), 3):
        x, y, conf = keypoints[i], keypoints[i+1], keypoints[i+2]

        # Eliminate low-confidence points.
        if conf > 0.2:
          pt = (int(x), int(y))
          cv2.circle(img, pt, radius=4, color=(0, 255, 0), thickness=-1)
          points.append(pt)
        else:
          points.append(None)

      # Draw the bones using lines joining point tuples from the Keypoints.
      for connection in HALPE_SKELETON:
        idx1, idx2 = connection
        # Check that both indices do not fall out of range.
        if idx1 < len(points) and idx2 < len(points):
          pt1 = points[idx1]
          pt2 = points[idx2]
          # Draw the line if and only if both points got detected.
          if pt1 is not None and pt2 is not None:
            cv2.line(img, pt1, pt2, color=(255, 0, 0), thickness=2)

    # Since OpenCV uses BGR but Imageio uses RGB, convert the colour channels.
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    writer.append_data(img_rgb)

  writer.close()
  print(f"{json_path} converted to a synthetic video: {output_path}")

In [ ]:
# Convert all JSON files in the Internal Dataset into synthetic MP4 videos..
gait_json_dir = Path("/content/drive/MyDrive/bachelor_thesis/internal_json")
gait_mp4_dir = Path("/content/drive/MyDrive/bachelor_thesis/internal_videos") # Save on Google Drive for future mounting.
gait_mp4_dir.mkdir(parents=True, exist_ok=True)

print(f"Videos found to process: {len(list(gait_json_dir.rglob("*.json")))}")

# Iterate through the JSON files.
for el in gait_json_dir.rglob("*.json"):
  relative_json_path = el.relative_to(gait_json_dir)
  relative_mp4_path = relative_json_path.with_suffix(".mp4") # Change suffix from .json to .mp4.
  drive_mp4_path = gait_mp4_dir / relative_mp4_path
  drive_mp4_path.parent.mkdir(parents=True, exist_ok=True)

  # If the JSON file has already been converted to a synthetic video, skip it.
  if drive_mp4_path.exists():
    continue

  print(f"Converting {el} into a synthetic video")
  json_to_video_v2(str(el), str(drive_mp4_path)) # Convert Path objects to strings.

print("Internal Dataset successfully rendered.")

## Dataframe Preparation

After downloading the two datasets, the next step consists of creating a **unified dataframe** that contains information about each video.

This dataframe will contain three fields:
1. `video_id`, which provides a unique identifier for each video.
2. `source`, which indicates whether the video comes from the Kaggle MMU Dataset or from the internal dataset.
3. `path`, which represents the path to the rendered video.
4. `width`, which denotes the width of each video frame.
5. `height`, which denotes the height of each video frame.
6. `parkinson`, which indicates whether the subject is healthy (`parkinson = 0`) or has Parkinson's disease (`parkinson = 1`).

In [ ]:
rows = []

# Add videos from the Kaggle MMU Dataset.
# Determine the labels for each class in the dataset.
kaggle_labels = {
    "normal": 0,
    "mild": 1,
    "moderate": 1,
    "severe": 1
}

kaggle_base_dir = "/content/drive/MyDrive/bachelor_thesis/kaggle_videos"

# Use os.walk to automate file search.
for root, dirs, files in os.walk(kaggle_base_dir):
  for file in files:
    if file.endswith(".mp4"):
      mp4_path = os.path.join(root, file)
      folder_name = os.path.basename(root).lower()

      if folder_name in kaggle_labels:
        video_id = os.path.splitext(file)[0]
        val = kaggle_labels[folder_name]

        rows.append({
            "video_id": video_id,
            "source": "Kaggle",
            "path": mp4_path,
            "width": 1280,
            "height": 720,
            "parkinson": val
        })

print("Kaggle MMU Dataset successfully extracted.")

# Add videos from the GAIT Dataset.
gait_base_dir = "/content/drive/MyDrive/bachelor_thesis/internal_videos"

# Use os.walk to automate file search.
for root, dirs, files in os.walk(gait_base_dir):
  for file in files:
    if file.endswith(".mp4"):
      video_path = os.path.join(root, file)
      # Use the video's relative path to create a unique ID.
      relative_path = os.path.relpath(video_path, gait_base_dir)
      video_id = os.path.splitext(relative_path)[0].replace(os.sep, "-")

      # Skip depth and ir videos to avoid dirty data.
      if "rgb" not in video_path:
        continue

      # Since all subjects in the GAIT Dataset are healthy, set parkinson = 0 for each.
      rows.append({
          "video_id": video_id,
          "source": "Internal",
          "path": video_path,
          "width": 848,
          "height": 480,
          "parkinson": 0
      })

print("Internal Dataset successfully extracted.")

# Aggregation and merging.
unified_df = pd.DataFrame(rows)

print(f"Total videos: {len(unified_df)}\n")

print("Count based on the source dataset:")
print(unified_df['source'].value_counts())
print("\nClass balance (0 = Healthy, 1 = Parkinson):")
print(unified_df['parkinson'].value_counts())

# Save the dataframe.
df_dir = "/content/drive/MyDrive/bachelor_thesis/dataframes"
os.makedirs(df_dir, exist_ok=True)
unified_df.to_csv("/content/drive/MyDrive/bachelor_thesis/dataframes/unified_dataset.csv", index=False)

## 2- Frame Extraction and Preprocessing

After the synthetic videos have been created and saved, the next step consists of extracting and cleaning the frames that will be used to train the model.

Since the synthetic videos are created based on files coming from different datasets, they might have different durations and the skeletons could be visible at different intervals.

For this reason, a **frame extraction interval** is defined depending on the video source and on the action performed by the subject.

- Since the videos in the Kaggle MMU Dataset are shorter and more consistent in the actions performed, the frames will be extracted **throughout the entire video**.
- Since the videos in the internal dataset have different lengths and show different actions, the following intervals are used:
  - For videos representing the `stairs_down` action, the frames will be extracted from **half of the video duration up to the end of the video**.
  - For videos representing the `stairs_up` action, the frames will be extracted **from the start of the video up to $75\%$ of the video duration**.
  - For videos representing the `walk` action, the frames will be extracted **from half of the video duration up to $70\%$ of the video duration**.

Additionally, these synthetic videos still present some **noise**, such as the presence of duplicated skeletons, requiring to **clean the videos and the extracted frames** in order to avoid training errors.

The frame extraction pipeline is implemented in the `frame_extraction` function, which **reads the input video and the corresponding keypoints file** and uses the **frame keys** contained in the keypoints file to **extract the video frames**.

Next, the function determines **which frames belong to the chosen frame extraction interval** and it **filters out all invalid frames** using the `frame_confidence` function, which reads the information associated to a frame in order to determine whether to keep it or discard it.

In particular, the function **discards** a frame under one of the following conditions:
- Keypoint information about the frame is **not found** in the file.
- **No skeleton** is detected in the frame.
- **Multiple skeletons** are detected in the frame.
- The **average confidence score** of the keypoints is **below the chosen threshold**.

After filtering out the invalid frames, the main function uses the `np.linspace()` function to **uniformly sample `num_frames` valid frames** from the extraction interval, eventually duplicating frames if less than `num_frames` frames are found.

Lastly, each sampled frame is processed by **converting the colour scheme from BGR to RGB, applying a median filter and resizing it to the target size**, eventually duplicating other frames or using a black placeholder frame whenever the extraction fails.

At the end of the pipeline, the function will return an array of shape $(num\_frames, 224, 224, 3)$.

In [ ]:
def frame_confidence(keypoints_input, frame_key, threshold=0.5):
  # This function checks whether the average keypoint confidence is above the chosen threshold.
  if isinstance(keypoints_input, str):
    # If the file path is given, open the corresponding file first.
    with open(keypoints_input, "r") as f:
      keypoints_data = json.load(f)
  else:
    keypoints_data = keypoints_input
  frame_info = None

  # JSON keypoints from the Kaggle Dataset are stored as lists.
  if isinstance(keypoints_data, list):
    clean_target = str(frame_key).split(".")[0] # Extract the frame ID regardless of whether the key is passed as x, "x" or "x.jpg".
    for item in keypoints_data:
      img_id = str(item.get("image_id", ""))
      if img_id == str(frame_key) or img_id.split(".")[0] == clean_target:
        frame_info = item
        break

  # JSON keypoints from the Internal Dataset are stored as nested dictionaries.
  elif isinstance(keypoints_data, dict):
    clean_target = str(frame_key).split(".")[0] # Extract the frame ID regardless of whether the key is passed as x, "x" or "x.jpg".
    possible_keys = [frame_key, str(frame_key), clean_target, f"{clean_target}.jpg"]
    for k in possible_keys:
      if k in keypoints_data:
        frame_info = keypoints_data[k]
        break

  if frame_info is None:
    # Fallback if the frame is not found.
    # print(f"Frame {frame_key} not found.") # Uncomment for debugging.
    return False

  # Extract information from the frame.
  bodies = []
  if isinstance(frame_info, dict):
    bodies = frame_info.get("bodies") or frame_info.get("people") or [frame_info]
  elif isinstance(frame_info, list):
    bodies = frame_info

  if not bodies:
    # Fallback if no body is detected.
    # print(f"No body was found in frame {frame_key}.") # Uncomment for debugging.
    return False
  elif len(bodies) != 1:
    # Fallback if more skeletons are detected in the frame.
    # print(f"Multiple skeletons detected in frame {frame_key}.") # Uncomment for debugging.
    return False

  # Compute the average confidence score and the number of visible keypoints for the frame.
  avg_confidence = -1.0 # Default value.
  body = bodies[0]

  keypoints = body.get("joints") or body.get("keypoints", [])
  if not keypoints:
    # Fallback in case no keypoints are found.
    return False

  scores = np.array(keypoints[2::3]) # Take just the confidence scores.
  if scores.size > 0 and scores.any():
    avg_confidence = max(avg_confidence, float(np.mean(scores)))

  return (avg_confidence > threshold)

In [ ]:
def frame_extraction_pipeline(video_path, keypoints_path, interval=(0.0, 1.0), target_size=(224, 224), threshold=0.5, num_frames=32):
  # Read the video and the keypoints.
  cap = cv2.VideoCapture(video_path)
  if not cap.isOpened():
    raise ValueError(f"Error when opening {video_path}.")

  total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

  with open(keypoints_path, "r") as f:
    keypoints_data = json.load(f)

  # Extract all the frame keys in the JSON file and convert them into sorted values.
  if isinstance(keypoints_data, list):
    # JSON keypoints from the Kaggle Dataset are stored as lists.
    raw_keys = [item.get("image_id") for item in keypoints_data if isinstance(item, dict) and "image_id" in item]
  elif isinstance(keypoints_data, dict):
    # JSON keypoints from the Internal Dataset are stored as nested dictionaries.
    raw_keys = list(keypoints_data.keys())
  else:
    raw_keys = []

  valid_frames = []
  for k in raw_keys:
    try:
      n = int(str(k).split(".")[0])
      valid_frames.append(n)
    except(ValueError, TypeError):
      # Fallback in case the key extraction fails.
      continue

  valid_frames = sorted(list(set(valid_frames))) # Eliminate duplicates and sort the frame keys.

  # Define the extraction interval.
  start_position = int(len(valid_frames) * interval[0])
  end_position = max(start_position, int(len(valid_frames) * interval[1]) - 1)
  interval_frames = valid_frames[start_position:end_position+1]

  # Filter out invalid or low-confidence frames, as well as frames presenting more than one skeleton.
  clean_valid_frames = [idx for idx in interval_frames if frame_confidence(keypoints_data, idx, threshold)]

  # Use np.linspace() to see which frames should be sampled.
  if clean_valid_frames:
    # Some valid frames have been found, so sample the positions and retrieve the corresponding frames.
    positions = np.linspace(0, len(clean_valid_frames) - 1, num=num_frames, dtype=int)
    frame_indices = [clean_valid_frames[p] for p in positions]
  else:
    # Fallback in case no valid frame has been found.
    print(f"No valid frame found for {video_path}.")
    frame_indices = []

  # Extract and clean the chosen frames.
  frames = []
  adjacent_frame = None # Buffer for duplicating corrupted/invalid frames.

  if frame_indices:
    # Iterate through the valid frames.
    for idx in frame_indices:
      cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
      ret, frame = cap.read()
      if ret and frame is not None:
        # The frame is valid and gets processed.
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) # Convert from OpenCV's BGR scheme to the RGB scheme.
        if "internal_videos" in video_path:
          # The frame_confidence() function does not eliminate all noise in these videos due to AlphaPose bugs in the rendering.
          rgb_frame = rgb_frame[:, int(0.4 * rgb_frame.shape[1]):, :]
        smoothed_frame = cv2.medianBlur(rgb_frame, 3) # Use OpenCV's native median filter.
        resized_frame = cv2.resize(smoothed_frame, target_size, interpolation=cv2.INTER_AREA)
        frames.append(resized_frame)
        adjacent_frame = resized_frame # Save the latest valid frame as a buffer.
      else:
        # The frame is not valid and gets replaced by the latest valid frame.
        frames.append(adjacent_frame)

    # Since invalid/corrupted frames might also appear at the start of the sequence, duplicate the first valid frames for those slots as well.
    to_duplicate = next((f for f in frames if f is not None), None) # Extract the first valid frame or get None if no valid frame exists.
    if to_duplicate is None:
      # Fallback in case no valid frame has been found.
      for i in range(num_frames):
        frames[i] = np.zeros((*target_size, 3), dtype=np.uint8)
    else:
      # Some valid frame to duplicate is found.
      for i in range(num_frames):
        if frames[i] is None:
          frames[i] = to_duplicate

  else:
    # Fallback in case no valid frame has been found.
    for _ in range(num_frames):
      zero_frame = np.zeros((*target_size, 3), dtype=np.uint8)
      frames.append(zero_frame)

  cap.release()

  frames_array = np.array(frames)
  print(f"{len(frames_array)} frames found for {video_path}. Shape: {frames_array.shape}")
  return frames_array

## Frame Storage

To avoid repeating the extraction procedure everytime, each array is **converted into a tensor** that **stored into a new dataframe containing a path to said tensor and the label associated to the original synthetic video**.

This step is useful as it will allow to load the ready tensors using just a custom `Dataset` class and the corresponding `DataLoader`.

In [ ]:
video_df = pd.read_csv("/content/drive/MyDrive/bachelor_thesis/dataframes/unified_dataset.csv")

for _, row in video_df.iterrows():
  # Extract the video path and recover the corresponding keypoints.
  video_path = row["path"]
  json_path = (video_path.replace("videos", "json")).replace(".mp4", ".json")

  # Create the tensor directories.
  tensor_path = (video_path.replace("videos", "tensors")).replace(".mp4", ".pt")
  os.makedirs(os.path.dirname(tensor_path), exist_ok=True)

  # Since the procedure may interrupt, skip any video whose frames have been previously extracted.
  if os.path.exists(tensor_path):
    continue

  # Frame extraction.
  print(f"Extracting frames from {video_path}.")
  if row["source"] == "Kaggle":
    # Kaggle Video frame extraction: Extract frames throughout the entire video and with a lower confidence threshold.
    frames = frame_extraction_pipeline(video_path, json_path, (0.0, 1.0), (224, 224), 0.15, 32)
  elif row["source"] == "Internal":
    # Internal Video frame extraction: Extract frames at different intervals and with a higher confidence threshold.
    if "stairs_down" in video_path:
      # Stairs down action.
      frames = frame_extraction_pipeline(video_path, json_path, (0.5, 1.0), (224, 224), 0.5, 32)
    elif "stairs_up" in video_path:
      frames = frame_extraction_pipeline(video_path, json_path, (0.0, 0.75), (224, 224), 0.5, 32)
    elif "walk" in video_path:
      frames = frame_extraction_pipeline(video_path, json_path, (0.5, 0.7), (224, 224), 0.5, 32)
    else:
      # Fallback for unknown actions.
      frames = None # Fallback placeholder value.
      print(f"Skipping unknown video {video_path}.")
  else:
    # Fallback for unknown videos.
    frames = None # Fallback placeholder value.
    print(f"Unknown source {row["source"]}: skipping {video_path}.")

  if frames is not None:
    # Convert the found frames from NumPy to PyTorch.
    print(f"Saving to {tensor_path}.")
    frame_tensor = torch.from_numpy(frames)
    torch.save(frame_tensor, tensor_path)

print("Frame extraction and storage successfully completed.")

In [ ]:
video_df = pd.read_csv("/content/drive/MyDrive/bachelor_thesis/dataframes/unified_dataset.csv")
tensors = []

for _, row in video_df.iterrows():
  # Extract the video path and recover the corresponding frame tensor.
  video_path = row["path"]
  tensor_path = (video_path.replace("videos", "tensors")).replace(".mp4", ".pt")

  # Skip videos for which no valid frame was found.
  tensor = torch.load(tensor_path)
  if torch.all(tensor == 0).item():
    print(f"No valid frames found for {video_path}: Skipping.")
    continue

  # Add the original video's ID and label, as well as the frame tensor, to the dataframe.
  tensors.append({
      "video_id": row["video_id"],
      "source": row["source"],
      "path": tensor_path,
      "parkinson": row["parkinson"]
  })

# Tensor dataframe aggregation.
tensor_df = pd.DataFrame(tensors)

print(f"Total tensors: {len(tensor_df)}.")

print("Count based on the source dataset:")
print(tensor_df['source'].value_counts())
print("\nClass balance (0 = Healthy, 1 = Parkinson):")
print(tensor_df['parkinson'].value_counts())

df_dir = "/content/drive/MyDrive/bachelor_thesis/dataframes"
os.makedirs(df_dir, exist_ok=True)
tensor_df.to_csv("/content/drive/MyDrive/bachelor_thesis/dataframes/tensor_dataset.csv", index=False)

print("Tensor dataframe successfully created.")

## Frame Visualization

After extracting frames from a video, it is possible to load the resulting tensor in order to look at the frames that have been extracted.

In [ ]:
# Load a sample tensor and look at its shape.
test = torch.load("/content/drive/MyDrive/bachelor_thesis/internal_tensors/dataset_blurred/FirstRun/Alessio/rgb/stairs_down/Alessio_stairs_down_1.pt")
print(f"Shape of the tensor: {test.shape}")

# Check whether the tensor contains black or discarded frames.
is_all_zero = torch.all(test == 0).item()
print(f"Is the tensor entirely composed by zeros? {is_all_zero}")

has_values = torch.any((test != 0) & ~torch.isnan(test)).item()
print(f"Are there valid values? {has_values}")

nan_frames = torch.isnan(test[:, 0, 0, 0]).sum().item() # Check just the first pixel.
print(f"Discarded frames: {nan_frames}.")

In [ ]:
def frame_visualization(tensor_file):
  # Load the tensor.
  frames = torch.load(tensor_file)
  n_frames = frames.shape[0]

  # Set grid dimensions.
  cols = 8
  rows = (n_frames + cols - 1) // cols

  # Create the main image and the subplots.
  fig, axes = plt.subplots(rows, cols, figsize=(20, 2.5 * rows))
  axes = axes.flatten()

  for i in range(n_frames):
    ax = axes[i]
    frame = frames[i].numpy()

    if np.isnan(frame).any():
      # If the frame was discarded, show a black image with a text.
      blk = np.zeros((224, 224, 3), dtype=np.uint8)
      ax.imshow(blk)
      ax.text(112, 112, "NaN", color="red", ha="center", va="center", fontsize=12, weight="bold")
    else:
      ax.imshow(frame.astype(np.uint8))

    # Add a title to each subplot to indicate the frame.
    ax.set_title(f"Frame {i}")
    ax.axis("off")

  for j in range(n_frames, len(axes)):
    axes[j].axis("off")

  plt.tight_layout()
  plt.show()

## 3 - Model Implementation

The model will be implemented from scratch based on the structure of [Arnab et al.'s *Video Vision Transformer*](https://arxiv.org/abs/2103.15691), which can be studied using the `VivitForVideoClassification` class provided by [Hugging Face](https://huggingface.co/docs/transformers/en/model_doc/vivit).

In [ ]:
# Analyse the structure of the Video Vision Transformer model.
model = VivitModel.from_pretrained("google/vivit-b-16x2-kinetics400", attn_implementation="sdpa", device_map="auto") # By default, this instance uses scaled dot-product attention and maps the model on available GPUs.
print(model)

In [ ]:
# Analyse the structure of the Video Vision Transformer model.
model = VivitForVideoClassification.from_pretrained("google/vivit-b-16x2-kinetics400", attn_implementation="sdpa", device_map="auto") # By default, this instance uses scaled dot-product attention and maps the model on available GPUs.
print(model)

### Tubelet Embeddings

The tubelet embedding method can be seen as an extension of the *Vision Transformer*'s embedding method for video data.

In fact, the idea is to **extract non-overlapping patches** of size $t \times h \times w$, where $t$ is known as the "tubelet size" while $(h, w)$ is the spatial resolution of each patch, and **linearly project** them to an embedding dimension $\mathbb{R}^d$ using a **3D convolution**.

Therefore, for a tensor of shape $(C, T, H, W)$, the number of extracted tokens will be equal to $N = \lfloor\frac{T}{t}\rfloor \cdot \lfloor\frac{H}{h}\rfloor \cdot \lfloor\frac{W}{w}\rfloor$.

**N.B.:** From now on, let $n_t = \lfloor\frac{T}{t}\rfloor$, $n_h = \lfloor\frac{H}{h}\rfloor$ and $n_w = \lfloor\frac{W}{w}\rfloor$.

In [ ]:
class VivitTubeletEmbeddings(nn.Module):
  def __init__(self, img_size=(224, 224), patch_size=(16, 16), num_frames=32, tubelet_size=2, in_channels=3, embed_dim=768):
    super().__init__()

    # Define the parameters, using the default values from the original paper.
    self.img_size = img_size
    self.patch_size = patch_size
    self.num_frames = num_frames
    self.tubelet_size = tubelet_size

    # Determine the spatial and temporal dimensions of each patch and determine the number of extracted tokens.
    self.num_spatial_patches = (img_size[0] // patch_size[0]) * (img_size[1] // patch_size[1]) # (H // h) * (W // w).
    self.num_temporal_patches = num_frames // tubelet_size # T // t.
    self.total_patches = self.num_spatial_patches * self.num_temporal_patches # N = (H // h) * (W // w) * (T // t)

    # Extract and project non-overlapping patches using a 3D convolution.
    self.proj = nn.Conv3d(
        in_channels=in_channels,
        out_channels=embed_dim,
        kernel_size=(tubelet_size, patch_size[0], patch_size[1]),
        stride=(tubelet_size, patch_size[0], patch_size[1])
    )

  def forward(self, x):
    # Take a tensor of shape (B, C, T, H, W) and project it to (B, embed_dim, T // t, H // h, W // w) using a 3D convolution.
    x = self.proj(x)

    # Flatten the tensor from (B, embed_dim, T // t, H // h, W // w) to (B, embed_dim, N), where N = (T // t) * (H // h) * (W // w).
    x = x.flatten(2) # This tells the block to flatten from dimension 2 onwards.

    # Reshape the tensor from (B, embed_dim, N) to (B, N, embed_dim) to make it compatible with the model.
    x = x.transpose(1, 2)

    return x

In [ ]:
class VivitEmbeddings(nn.Module):
  def __init__(self, img_size=(224, 224), patch_size=(16, 16), num_frames=32, tubelet_size=2, in_channels=3, embed_dim=768, dropout=0.0):
    super().__init__()
    self.emb = VivitTubeletEmbeddings(img_size=img_size, patch_size=patch_size, num_frames=num_frames, tubelet_size=tubelet_size, in_channels=in_channels, embed_dim=embed_dim)
    self.drop = nn.Dropout(dropout)

  def forward(self, x):
    # Apply the tubelet embedding mechanism.
    x = self.emb(x)

    # Apply the dropout mask.
    x = self.drop(x)

    return x

### Transformer Encoder Block

A single encoder block alternates between **computing self-attention** and a **multi-layer perceptron**.
1. On an input $\mathbf{X} \in \mathbb{R}^{N \times d}$, the self-attention mechanism extracts the queries $\mathbf{Q} = \mathbf{XW}_q$, keys $\mathbf{K} = \mathbf{XW}_k$ and values $\mathbf{V} = \mathbf{XW}_v$ by a linear projection on the input and uses these vectors to compute the attention scores
$$
\mathbf{Y} = \text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Softmax}\bigg(\frac{\mathbf{QK}^T}{\sqrt{d_k}}\bigg)\mathbf{V}
$$
2. After computing the attention scores, the sequence is passed into a **multilayer perceptron** whose structure follows the one implemented in the *Vision Transformer* backbone used for the original model.

In [ ]:
class VivitAttention(nn.Module):
  def __init__(self, num_spatial_patches, num_temporal_patches, embed_dim=768, num_heads=12, qkv_bias=True, dropout=0.0):
    super().__init__()

    # Define the parameters.
    self.num_spatial_patches = num_spatial_patches
    self.num_temporal_patches = num_temporal_patches
    self.num_heads = num_heads
    self.head_dim = embed_dim // num_heads # Defined to simplify computations in the forward method.

    # Extract queries, keys and values using a linear projection with additive bias.
    self.q_proj = nn.Linear(embed_dim, embed_dim, bias=qkv_bias)
    self.k_proj = nn.Linear(embed_dim, embed_dim, bias=qkv_bias)
    self.v_proj = nn.Linear(embed_dim, embed_dim, bias=qkv_bias)
    self.attn_drop = nn.Dropout(dropout)
    self.proj = nn.Linear(embed_dim, embed_dim, bias=qkv_bias)
    self.proj_drop = nn.Dropout(dropout)

  def forward(self, x, num_spatial_patches, num_temporal_patches):
    B, N, D = x.shape # Note: N = 1 + (num_spatial_patches * num_temporal_patches), D = embed_dim.

    q = self.q_proj(x).reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3) # Compute the queries.
    k = self.k_proj(x).reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3) # Compute the keys.
    v = self.v_proj(x).reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3) # Compute the values.

    x = F.scaled_dot_product_attention(q, k, v)
    x = x.transpose(1, 2).reshape(B, N, D)
    x = self.proj(x)
    x = self.proj_drop(x)

    return x

In [ ]:
class VivitMLP(nn.Module):
  def __init__(self, in_features, hidden_features, dropout=0.0):
    super().__init__()
    self.fc1 = nn.Linear(in_features, hidden_features) # Project from in_features to hidden_features.
    # self.act = nn.GELU(approximate="tanh") # Set approximate="tanh" for the FastGELU approximation.
    self.fc2 = nn.Linear(hidden_features, in_features) # Project from hidden_features to in_features.
    self.drop = nn.Dropout(dropout)

  def forward(self, x):
    x = self.fc1(x)
    x = 0.5 * x * (1.0 + torch.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3.0))))
    x = self.drop(x)
    x = self.fc2(x)
    x = self.drop(x)
    return x

In [ ]:
class VivitLayer(nn.Module):
  def __init__(self, num_spatial_patches, num_temporal_patches, embed_dim, num_heads, mlp_ratio=4.0, qkv_bias=True, dropout=0.0):
    super().__init__()

    # Define the parameters.
    self.num_spatial_patches = num_spatial_patches
    self.num_temporal_patches = num_temporal_patches

    # Apply layer normalization and compute attention scores.
    self.norm1 = nn.LayerNorm(embed_dim, eps=1e-6, elementwise_affine=True) # Use the settings indicated in the original model.
    self.attn = VivitAttention(self.num_spatial_patches, self.num_temporal_patches, embed_dim, num_heads=num_heads, qkv_bias=qkv_bias, dropout=dropout)
    self.norm2 = nn.LayerNorm(embed_dim, eps=1e-6, elementwise_affine=True) # Use the settings indicated in the original model.

    # Pass through the multilayer perceptron.
    self.hidden_features = int(embed_dim * mlp_ratio) # Chosen according to the original backbone.
    self.mlp = VivitMLP(in_features=embed_dim, hidden_features=self.hidden_features, dropout=dropout)
    self.drop = nn.Dropout(dropout)

  def forward(self, x, num_spatial_patches, num_temporal_patches):
    x = x + self.attn(self.norm1(x), num_spatial_patches, num_temporal_patches) # x = x + MSA(LN(x)).
    x = x + self.mlp(self.norm2(x)) # x = x + MLP(LN(x)).
    return x

### Adapting the Classification Head

While the original *Video Vision Transformer* backbone was implemented for multiclass classification, this new model implements **binary classification for anomaly detection**, requiring to adjust the classification head for this new task.

For this reason, the `AnomalyHead` class simply features a linear layer that upscales the features before processing them with an activation function, ultimately projecting the result onto a one-dimensional logit.

In [ ]:
class AnomalyHead(nn.Module):
  def __init__(self, in_features, hidden_features):
    super().__init__()

    self.fc1 = nn.Linear(in_features, hidden_features)
    self.act = nn.ReLU() # Used in Versions 1-3 of the model.
    # self.act = nn.GELU() # Used in Versions 4-6 of the model.
    self.fc2 = nn.Linear(hidden_features, 1)

  def forward(self, x):
    x = self.fc1(x)
    x = self.act(x)
    logit = self.fc2(x)
    return logit

### The Complete Model

The overall structure of the model will be closely based on the original *Video Vision Transformer* architecture used for the `VivitForVideoClassificationClass`.

In fact, the model first extracts non-overlapping patch embeddings from the input tensor and later prepends the `[CLS]` token to the embedding sequence, also adding the positional embeddings.

Next, the model passes the sequence to the transformer encoder, which consists of 12 repeated blocks in accordance to the original architecture.

Lastly, the model applies layer normalization and extracts the representation of the `[CLS]` token, which will be used by the final classification head to compute the logit for the input tensor.

In [ ]:
class GaitViViT(nn.Module):
  def __init__(self, img_size=(224, 224), patch_size=(16, 16), num_frames=32, tubelet_size=2, in_channels=3, embed_dim=768, depth=12, num_heads=12, mlp_ratio=4.0, qkv_bias=True, cls_ratio=2.0, dropout=0.0):
    super().__init__()

    # Extract the embeddings and determine the number of spatial and temporal batches.
    self.patch_embed = VivitEmbeddings(
        img_size=img_size,
        patch_size=patch_size,
        num_frames=num_frames,
        tubelet_size=tubelet_size,
        in_channels=in_channels,
        embed_dim=embed_dim,
        dropout=dropout
        )
    self.num_spatial_patches = self.patch_embed.emb.num_spatial_patches
    self.num_temporal_patches = self.patch_embed.emb.num_temporal_patches

    # Create and initialize the [CLS] token and the patch embeddings.
    self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
    self.pos_embed = nn.Parameter(torch.zeros(1, 1 + self.num_spatial_patches * self.num_temporal_patches, embed_dim))
    nn.init.normal_(self.cls_token, std=0.02) # Initialize from a Gaussian distribution with mean = 0 and standard deviation = 0.02.
    nn.init.normal_(self.pos_embed, std=0.02) # Initialize from a Gaussian distribution with mean = 0 and standard deviation = 0.02.

    self.pos_drop = nn.Dropout(dropout)

    # Create the transformer encoder block using the VivitLayer class.
    self.blocks = nn.ModuleList([
        VivitLayer(
            num_spatial_patches=self.num_spatial_patches,
            num_temporal_patches=self.num_temporal_patches,
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            qkv_bias=qkv_bias,
            dropout=dropout
        )
        for _ in range(depth)
    ])

    # Apply layer normalization before entering the classification head.
    self.norm = nn.LayerNorm(embed_dim, eps=1e-6, elementwise_affine=True) # Use the settings indicated in the original model.

    # Enter the classification head.
    self.head_hf = int(embed_dim * cls_ratio)
    self.head = AnomalyHead(in_features=embed_dim, hidden_features=self.head_hf)

  def forward(self, x):
    # Determine the batch size.
    B = x.shape[0] # Remember that x will have shape (B, C, T, H, W)

    # Extract patch embeddings.
    x = self.patch_embed(x)

    # Prepend the classification token to the patch sequence and add the positional embeddings.
    cls_tokens = self.cls_token.expand((B, -1, -1)) # Expand from (1, 1, embed_dim) to (B, 1, embed_dim), with -1 indicating not to change the size of that dimension.
    x = torch.cat([cls_tokens, x], dim=1) # Concatenate to shape (B, 1 + num_spatial_patches * num_temporal_patches, embed_dim).
    x = x + self.pos_embed
    x = self.pos_drop(x)

    # Pass through the transformer encoder.
    for block in self.blocks:
      x = block(x, self.num_spatial_patches, self.num_temporal_patches) # Remember that the forward function of the block requires specifying the number of patches as well.

    x = self.norm(x)

    # Extract the classification token that will be passed to the classification head.
    cls_output = x[:, 0, :] # Collapse to shape (B, embed_dim).
    logit = self.head(cls_output)

    return logit

### Model Initialization

After creating an instance of the model, it is possible to **load the pre-trained weights for the backbone**.

However, since weight names might differ, this procedure requires **mapping these weights onto the corresponding weights in the custom model**.

In [ ]:
def weight_map(hf_state_dict):
  new_state_dict = {}

  # Map weights for the [CLS] token and the positional embeddings.
  new_state_dict["cls_token"] = hf_state_dict["vivit.embeddings.cls_token"]
  new_state_dict["pos_embed"] = hf_state_dict["vivit.embeddings.position_embeddings"]

  # Map weights for the patch embeddings.
  new_state_dict["patch_embed.emb.proj.weight"] = hf_state_dict["vivit.embeddings.patch_embeddings.projection.weight"]
  new_state_dict["patch_embed.emb.proj.bias"] = hf_state_dict["vivit.embeddings.patch_embeddings.projection.bias"]

  # Iteratively map weights for each layer.
  layer_indices = set()
  for k in hf_state_dict.keys():
    if k.startswith("vivit.layers."):
      # These keys have names of type "vivit.layers.{idx}.{component}.{weight/bias}".
      idx = int(k.split(".")[2])
      layer_indices.add(idx)

  for i in sorted(layer_indices):
    hf_prefix = f"vivit.layers.{i}." # Keys in the original model start with "vivit.layers.{i}".
    my_prefix = f"blocks.{i}." # Keys in the GaitViViT model start with "blocks.{i}".

    # Map weights for queries, keys and values.
    new_state_dict[f"{my_prefix}attn.q_proj.weight"] = hf_state_dict[f"{hf_prefix}attention.q_proj.weight"]
    new_state_dict[f"{my_prefix}attn.k_proj.weight"] = hf_state_dict[f"{hf_prefix}attention.k_proj.weight"]
    new_state_dict[f"{my_prefix}attn.v_proj.weight"] = hf_state_dict[f"{hf_prefix}attention.v_proj.weight"]

    if f"{hf_prefix}attention.q_proj.bias" in hf_state_dict:
      # Map biases if and only if they are defined for the GaitViViT model.
      new_state_dict[f"{my_prefix}attn.q_proj.bias"] = hf_state_dict[f"{hf_prefix}attention.q_proj.bias"]
      new_state_dict[f"{my_prefix}attn.k_proj.bias"] = hf_state_dict[f"{hf_prefix}attention.k_proj.bias"]
      new_state_dict[f"{my_prefix}attn.v_proj.bias"] = hf_state_dict[f"{hf_prefix}attention.v_proj.bias"]

    # Map weights for the attention output.
    new_state_dict[f"{my_prefix}attn.proj.weight"] = hf_state_dict[f"{hf_prefix}attention.o_proj.weight"]
    new_state_dict[f"{my_prefix}attn.proj.bias"] = hf_state_dict[f"{hf_prefix}attention.o_proj.bias"]

    # Map weights for layer normalization.
    new_state_dict[f"{my_prefix}norm1.weight"] = hf_state_dict[f"{hf_prefix}layernorm_before.weight"]
    new_state_dict[f"{my_prefix}norm1.bias"] = hf_state_dict[f"{hf_prefix}layernorm_before.bias"]
    new_state_dict[f"{my_prefix}norm2.weight"] = hf_state_dict[f"{hf_prefix}layernorm_after.weight"]
    new_state_dict[f"{my_prefix}norm2.bias"] = hf_state_dict[f"{hf_prefix}layernorm_after.bias"]

    # Map weights for the VivitMLP component.
    new_state_dict[f"{my_prefix}mlp.fc1.weight"] = hf_state_dict[f"{hf_prefix}mlp.fc1.weight"]
    new_state_dict[f"{my_prefix}mlp.fc1.bias"] = hf_state_dict[f"{hf_prefix}mlp.fc1.bias"]
    new_state_dict[f"{my_prefix}mlp.fc2.weight"] = hf_state_dict[f"{hf_prefix}mlp.fc2.weight"]
    new_state_dict[f"{my_prefix}mlp.fc2.bias"] = hf_state_dict[f"{hf_prefix}mlp.fc2.bias"]

  # Map weights for the final layer normalization before the classification head.
  new_state_dict["norm.weight"] = hf_state_dict["vivit.layernorm.weight"]
  new_state_dict["norm.bias"] = hf_state_dict["vivit.layernorm.bias"]

  # Note that head weights will not be mapped as the new classification head will be trained from scratch.

  return new_state_dict

**N.B.:** Further analysis revealed slight discrepancies of the order of $10^{-8}$ in the weight mapping procedure, although this is likely due to hardware limitations when working with floating-point arithmetic.

## 4 - Training

Since the goal of this project is to adapt the *Video Vision Transformer* to a supervised anomaly detection task, the model will be trained using a hybrid **transfer learning** approach that combines linear probing and model fine-tuning.

### Data Split

The dataset will be split into a **training dataset**, a **validation dataset** and a **testing dataset**.

| Subset | Size |
| :---: | :---:|
| Training | $70\%$ |
| Validation | $10\%$ |
| Testing | $20\%$ |

To avoid data leakage, the training, validation and testing datasets are constructed using the `StratifiedKFold` function, which combines **$K$-fold cross-validation** and **stratified sampling** to ensure representative splits across all folds.

In [ ]:
def split_data(df):
  # In order to avoid data leakage, extract patient IDs and the respective diagnosis.
  patients = df[["patient_id", "parkinson"]].drop_duplicates().reset_index(drop=True)
  X_p = patients["patient_id"].values
  y_p = patients["parkinson"].values
  fold_indices = {}

  # Outer split: 20% is isolated as the testing set, the remaining 80% will be used to build the training and validation set.
  outer_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

  for i, (train_val_p_idx, test_p_idx) in enumerate(outer_split.split(X_p, y_p)):
    # Determine which patients belong to the testing set and which patients will belong to the training or validation set.
    train_val_patients = X_p[train_val_p_idx]
    test_patients = X_p[test_p_idx]
    y_train_val_p = y_p[train_val_p_idx]

    # Inner split: Around 10% of the original dataset is isolated as the validation set, the rest will compose the training set.
    inner_split = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)
    train_p_idx, val_p_idx = next(inner_split.split(train_val_patients, y_train_val_p))
    train_patients = train_val_patients[train_p_idx]
    val_patients = train_val_patients[val_p_idx]

    # Determine the indices of the training, validation and testing sets.
    train_idx = df.index[df["patient_id"].isin(train_patients)].tolist()
    val_idx = df.index[df["patient_id"].isin(val_patients)].tolist()
    test_idx = df.index[df["patient_id"].isin(test_patients)].tolist()

    # Save the indices for the current fold.
    fold_indices[i] = {"train": train_idx, "val": val_idx, "test": test_idx}

  return fold_indices

In [ ]:
# Load the dataframe and determine the splits.
data = pd.read_csv("/content/drive/MyDrive/bachelor_thesis/dataframes/tensor_dataset.csv")
folds = split_data(data)

### Training Loop

Each epoch of the training loop consists of a **training step**, where the model performs a forward pass and uses the results to adjust its weights during the backward pass, and a **validation step**, where the model performs an unbiased forward pass.

**N.B.:** To avoid computational overhead, training is performed using gradient accumulation and mixed precision arithmetic.

In [ ]:
def get_metrics(y_true, y_probs, threshold=0.5):
  y_preds = (y_probs >= threshold).astype(int) # Convert from boolean to integer.

  # Compute each metric.
  accuracy = accuracy_score(y_true, y_preds)
  precision = precision_score(y_true, y_preds, zero_division=0)
  recall = recall_score(y_true, y_preds, zero_division=0)
  f1 = f1_score(y_true, y_preds, zero_division=0)
  try:
    roc_auc = roc_auc_score(y_true, y_probs)
  except ValueError:
    roc_auc = 0.0

  return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1, "roc_auc": roc_auc}

In [ ]:
def training_loop(model, dataloader, loss_function, optimizer, device, accumulation_steps=2):
  model.train() # Set the model in training mode.
  running_loss = 0.0
  all_targets, all_probs = [], [] # Needed to compute metrics.

  # Reset gradients.
  optimizer.zero_grad()

  for batch_idx, (inputs, targets) in enumerate(dataloader):
    # Move data to the chosen device.
    inputs, targets = inputs.to(device), targets.float().to(device) # Labels are converted to float for compatibility with BCEWithLogitsLoss.

    # Run a forward pass using mixed precision.
    with autocast("cuda"):
      logits = model(inputs).squeeze(-1)
      loss = loss_function(logits, targets)
      loss = loss / accumulation_steps
    scaler.scale(loss).backward()

    # Run the backward pass whenever needed.
    if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(dataloader):
      scaler.step(optimizer)
      scaler.update()
      optimizer.zero_grad()

    # Update the running loss and save data to get metrics.
    running_loss += (loss.item() * accumulation_steps) * inputs.size(0)
    probs = torch.sigmoid(logits).detach().cpu().numpy()
    all_probs.extend(probs)
    all_targets.extend(targets.detach().cpu().numpy())

  epoch_loss = running_loss / len(dataloader.dataset) # Average across batch losses.
  metrics = get_metrics(np.array(all_targets), np.array(all_probs))
  metrics["loss"] = epoch_loss

  return metrics

In [ ]:
def validation_loop(model, dataloader, loss_function, device, threshold=0.5, return_raw=False):
  model.eval() # Set the model in evaluation mode, disabling stochastic patterns.
  running_loss = 0.0
  all_targets, all_probs = [], [] # Needed to compute metrics.

  with torch.no_grad():
    for inputs, targets in dataloader:
      # Move data to the chosen device.
      inputs, targets = inputs.to(device), targets.float().to(device) # Labels are converted to float for compatibility with BCEWithLogitsLoss.

      # Run a forward pass using mixed precision.
      with autocast("cuda"):
        logits = model(inputs).squeeze(-1)
        loss = loss_function(logits, targets)

      # Update the running loss and save data to get metrics.
      running_loss += loss.item() * inputs.size(0)
      probs = torch.sigmoid(logits).detach().cpu().numpy()
      all_probs.extend(probs)
      all_targets.extend(targets.detach().cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset) # Average across batch losses.
    metrics = get_metrics(np.array(all_targets), np.array(all_probs), threshold=threshold)
    metrics["loss"] = epoch_loss

  if return_raw:
    return metrics, np.array(all_targets), np.array(all_probs)
  else:
    return metrics

### Linear Probing

The first step of the training procedure is the **linear probing** phase, which consists of freezing the backbone and training just the final classification head.

This step lasts **5 epochs** and the model is trained using **stochastic gradient descent with momentum** and **cosine annealing scheduling with linear warmups**.

In [ ]:
def linear_probing(model, dataloaders, loss_function, num_epochs=5, model_path="/content/drive/MyDrive/bachelor_thesis/models/best_head.pth", metrics_path="/content/drive/MyDrive/bachelor_thesis/metrics/head_lp.csv", device="cpu", checkpoint_path="/content/drive/MyDrive/bachelor_thesis/models/lp_resume_checkpoint.pth"):
  lp_history = [] # To save the metrics.
  best_f1 = 0.0 # Due to class imbalance, F1-score tends to be a more reliable metric.

  # Freeze the backbone to train just the head.
  for name, param in model.named_parameters():
    if "head" not in name:
      param.requires_grad = False
    else:
      param.requires_grad = True

  # Define the optimizer and the scheduler.
  optimizer_lp = torch.optim.SGD(model.head.parameters(), lr=3e-04, momentum=0.9) # Use a higher learning rate for this stage.
  # scheduler_lp = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_lp, T_max=num_epochs, eta_min=1e-5)
  training_steps = (len(dataloaders["train"]) * num_epochs) // 2 # 2 is the number of gradient accumulation steps.
  scheduler_lp = get_cosine_schedule_with_warmup(optimizer_lp, num_warmup_steps = int(0.1 * training_steps), num_training_steps=training_steps) # Apply warmup for 10% of the training.

  # Recover the latest checkpoint.
  start_epoch = 0
  if os.path.exists(checkpoint_path):
    print("Resuming Linear Probing.")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer_lp.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler_lp.load_state_dict(checkpoint["scheduler_state_dict"])
    best_f1 = checkpoint["best_f1"]
    lp_history = checkpoint["lp_history"]
    start_epoch = checkpoint["epoch"]

  # Run training and validation through each epoch.
  for epoch in range(start_epoch, num_epochs):
    # Get the training and validation metrics.
    train_metrics = training_loop(model, dataloaders["train"], loss_function, optimizer_lp, device)
    val_metrics = validation_loop(model, dataloaders["val"], loss_function, device)

    # Update the learning rate for next epoch.
    scheduler_lp.step()

    # Save the weights whenever the validation F1-score reaches a new global maximum.
    if val_metrics["f1"] > best_f1:
      best_f1 = val_metrics["f1"]
      torch.save(model.state_dict(), model_path)

    # Save the metrics for the current epoch.
    row = {"epoch": epoch + 1, "phase": "lp", "lr": scheduler_lp.get_last_lr()[0]}
    row.update({f"train_{k}": v for k, v in train_metrics.items()})
    row.update({f"val_{k}": v for k, v in val_metrics.items()})
    lp_history.append(row)

    # Save the checkpoint.
    torch.save({"epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer_lp.state_dict(),
                "scheduler_state_dict": scheduler_lp.state_dict(),
                "best_f1": best_f1,
                "lp_history": lp_history}, checkpoint_path)

  # Save the complete metrics.
  df_lp = pd.DataFrame(lp_history)
  df_lp.to_csv(metrics_path, index=False)

  return model

### Full Finetuning

After the linear probing phase, the **fine-tuning** phase unfreezes the entire model to adapt it to the new task.

This step lasts **15 epochs** and the model is trained using **stochastic gradient descent with momentum and weight decay** and **cosine annealing scheduling**.

Since fine-tuning tends to be computationally expensive and can lead to overfitting, the model is trained using **grid search**, testing different combinations of starting learning rate and weight decay, and introducing an **early stopping counter** before the model starts overfitting.

In [ ]:
def model_finetuning(model, dataloaders, loss_function, num_epochs=15, head_path="/content/drive/MyDrive/bachelor_thesis/models/best_head.pth", model_path="/content/drive/MyDrive/bachelor_thesis/models/best_model.pth", metrics_path="/content/drive/MyDrive/bachelor_thesis/metrics/model_ft.csv", device="cpu", checkpoint_path="/content/drive/MyDrive/bachelor_thesis/models/ft_resume_checkpoint.pth"):
  ft_history = [] # To save the metrics.
  global_best_f1 = 0.0 # Due to class imbalance, F1-score tends to be a more reliable metric.

  # Define the parameter grid, focusing on the starting learning rate and the weight decay.
  parameters = {
      "start_lr": [1e-6, 1e-5, 1e-4],
      "weight_decay": [1e-2, 0.1]
  }
  grid = ParameterGrid(parameters)

  # Recover the latest checkpoint.
  start_run_idx = 0 # To keep track of the hyperparameter combinations.
  start_epoch = 0 # To keep track of the current epoch.
  if os.path.exists(checkpoint_path):
    print("Resuming Finetuning.")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    start_run_idx = checkpoint["run_idx"]
    start_epoch = checkpoint["epoch"]
    ft_history = checkpoint["ft_history"]
    global_best_f1 = checkpoint["global_best_f1"]

  # Iterate through the parameter configurations.
  for idx, params in enumerate(grid):
    if idx < start_run_idx:
      # Skip completed runs.
      continue

    print(f"Run {idx + 1} | Parameters: {params}")

    # Define the parameters for early stopping.
    patience = 5
    tolerance = 0.001

    # Determine whether to resume the current run or reset everything for a new run.
    if idx == start_run_idx and os.path.exists(checkpoint_path):
      # Resume the current run: Load the latest model and the most recent metrics for the run.
      print(f"Resuming Run {idx + 1}.")
      checkpoint = torch.load(checkpoint_path, map_location=device)
      model.load_state_dict(checkpoint["model_state_dict"])
      run_best_f1 = checkpoint["run_best_f1"]
      counter = checkpoint["counter"]
      current_start_epoch = start_epoch
    elif idx > start_run_idx or not os.path.exists(checkpoint_path):
      # Reset for a new run: Load the weights of the head and unfreeze the backbone, resetting the run.
      model.load_state_dict(torch.load(head_path, map_location=device))
      for param in model.parameters():
        param.requires_grad = True
      run_best_f1 = 0.0
      counter = 0
      current_start_epoch = 0

    # Define the optimizer and the scheduler.
    optimizer_ft = torch.optim.SGD(model.parameters(), lr=params["start_lr"], momentum=0.9, weight_decay=params["weight_decay"])
    scheduler_ft = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=num_epochs, eta_min=1e-7)
    if idx == start_run_idx and os.path.exists(checkpoint_path):
      # Resume the current run: Load the latest optimizer and scheduler states.
      optimizer_ft.load_state_dict(checkpoint["optimizer_state_dict"])
      scheduler_ft.load_state_dict(checkpoint["scheduler_state_dict"])

    # Run training and validation through each epoch.
    for epoch in range(current_start_epoch, num_epochs):
      # Get the training and validation metrics.
      train_metrics = training_loop(model, dataloaders["train"], loss_function, optimizer_ft, device)
      val_metrics = validation_loop(model, dataloaders["val"], loss_function, device)

      # Update the learning rate for the next epoch.
      scheduler_ft.step()

      # Save the metrics for the current epoch.
      row = {"epoch": epoch + 1, "phase": "ft", "lr": scheduler_ft.get_last_lr()[0]}
      row.update(params)
      row.update({f"train_{k}": v for k, v in train_metrics.items()})
      row.update({f"val_{k}": v for k, v in val_metrics.items()})
      ft_history.append(row)

      current_f1 = val_metrics["f1"]

      # Save the weights whenever the validation F1-score reaches a new global maximum.
      if current_f1 > global_best_f1:
        global_best_f1 = current_f1
        torch.save(model.state_dict(), model_path)

      # Check for early stopping.
      if current_f1 > run_best_f1 + tolerance:
        # Significant improvement: Reset the counter.
        run_best_f1 = current_f1
        counter = 0
      else:
        # No significant improvement: Update the counter.
        counter += 1

      # Save the checkpoint.
      torch.save({"run_idx": idx,
                  "epoch": epoch + 1,
                  "model_state_dict": model.state_dict(),
                  "optimizer_state_dict": optimizer_ft.state_dict(),
                  "scheduler_state_dict": scheduler_ft.state_dict(),
                  "global_best_f1": global_best_f1,
                  "run_best_f1": run_best_f1,
                  "ft_history": ft_history,
                  "counter": counter}, checkpoint_path)

      # Check for early stopping.
      if counter >= patience:
        print(f"Early stopping triggered at epoch {epoch + 1} for parameters {params}.")
        break

  # Save the complete metrics.
  df_ft = pd.DataFrame(ft_history)
  df_ft.to_csv(metrics_path, index=False)

  return model

### Testing Stage

After the linear probing and fine-tuning steps, the **best-performing model** performs the testing stage in order to assess its learning ability on unseen data.

In [ ]:
def save_confusion_matrix(y_true, y_probs, threshold=0.5, save_path="/content/drive/MyDrive/bachelor_thesis/metrics/confusion_matrix.png"):
  # Convert logits into predictions.
  y_preds = (y_probs >= threshold).astype(int)

  # Compute and plot the confusion matrix.
  cm = confusion_matrix(y_true, y_preds)
  plt.figure(figsize=(8, 6))
  sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
              xticklabels=['Healthy', 'Parkinson'],
              yticklabels=['Healthy', 'Parkinson'])

  plt.title(f'Confusion Matrix')
  plt.xlabel('Predicted Label')
  plt.ylabel('True Label')

  # Save the image.
  plt.savefig(save_path, dpi=300, bbox_inches='tight')
  plt.close()

In [ ]:
def model_test(model, dataloaders, loss_function, best_model_path="/content/drive/MyDrive/bachelor_thesis/models/best_model.pth", metrics_path="/content/drive/MyDrive/bachelor_thesis/metrics/model_test.csv", matrix_path="/content/drive/MyDrive/bachelor_thesis/metrics/model_test_cm.png", device="cpu"):
  # Load the best model and run a test loop.
  model.load_state_dict(torch.load(best_model_path, map_location=device))
  test_metrics, y_true, y_probs = validation_loop(model, dataloaders["test"], loss_function, device, threshold=0.5, return_raw=True)

  # Save the metrics.
  data = []
  data.append(test_metrics)
  df = pd.DataFrame(data)
  df.to_csv(metrics_path, index=False)

  # Get the confusion matrix.
  save_confusion_matrix(y_true, y_probs, threshold=0.5, save_path=matrix_path)

  # Print the results.
  for k, v in test_metrics.items():
    print(f"Test {k}: {v}.")

  return test_metrics, y_true, y_probs

### Ablation Study

Due to the medical purpose of the model, an ablation study is carried out in order to understand which classification threshold would be more suitable for this task.

| Threshold | Advantages | Disadvantages |
| :---: | :---: | :---: |
| Below $0.5$ | The model is more likely to detect early/mild symptoms | The model is vulnerable to false positives |
| Above $0.5$ | The model is more robust against false positives | The model might overlook early/mild symptoms |

In [ ]:
def ablation_study(y_true, y_probs, thresholds=np.arange(0.1, 1.0, 0.1), csv_path="/content/drive/MyDrive/bachelor_thesis/metrics/threshold_ablation.csv", plot_path="/content/drive/MyDrive/bachelor_thesis/metrics/threshold_plot.png"):
  ablation_results = []

  # Run the ablation study.
  for t in thresholds:
    metrics = get_metrics(y_true, y_probs, threshold=t)
    metrics["threshold"] = t
    ablation_results.append(metrics)

  ablation_df = pd.DataFrame(ablation_results)
  ablation_df.to_csv(csv_path, index=False)

  # Create a plot for the ablation study.
  plt.figure(figsize=(10, 6))
  sns.set_style("whitegrid")

  # Draw the metrics.
  plt.plot(ablation_df["threshold"], ablation_df["f1"], marker='o', label="F1-Score", color="green")
  plt.plot(ablation_df["threshold"], ablation_df["precision"], marker='s', label="Precision", color="blue")
  plt.plot(ablation_df["threshold"], ablation_df["recall"], marker='^', label="Recall", color="orange")
  plt.axhline(y=ablation_df["roc_auc"].iloc[0], color='r', linestyle=':', label=f'ROC-AUC ({ablation_df["roc_auc"].iloc[0]:.3f})')

  # Add the final details.
  plt.title("Ablation Study: Performance vs Threshold")
  plt.xlabel("Classification Threshold")
  plt.ylabel("Score")
  plt.legend(loc="lower center")
  plt.savefig(plot_path, dpi=300)

  return ablation_df

### Main Loop

Overall, the model is trained using $K$-fold cross-validation, which comes in handy to obtain more reliable results, and taking binary cross-entropy loss in order to address the dataset's class imbalance.

For each generated fold, the **initialized model undergoes linear probing and fine-tuning**, saving the best performing model, and, after the first two steps, the best-performing model is used to perform the **testing stage** and the **ablation study**.

**N.B.:** The following loop uses a generic default configuration for saving models and metrics.

In [ ]:
# Choose which device to use for the training procedure.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}.")

# Define the loss function.
weight = 947 / 115 # Weights taken from the class distribution in tensor_dataset.csv.
pos_weight = torch.tensor([weight]).to(device)
loss_function = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Create, if needed, the directories to store model weights and metrics.
os.makedirs("/content/drive/MyDrive/bachelor_thesis/models", exist_ok=True)
os.makedirs("/content/drive/MyDrive/bachelor_thesis/metrics", exist_ok=True)

In [ ]:
# Iterate through folds.
for k, v in folds.items():
  # Since the procedure may interrupt, skip any completed folds.
  final_ablation_path = f"/content/drive/MyDrive/bachelor_thesis/metrics/ablation_study_fold_{k + 1}.csv"
  if os.path.exists(final_ablation_path):
    print(f"Fold {k + 1} previously completed. Skipping to the next one.")
    continue

  # Determine the splits.
  train_idx, val_idx, test_idx = v["train"], v["val"], v["test"]
  train_data = data.iloc[train_idx].copy()
  val_data = data.iloc[val_idx].copy()
  test_data = data.iloc[test_idx].copy()

  # Create the datasets and the corresponding dataloaders.
  train_dataset = GaitViViTDataset(tensor_df=train_data, frames_per_video=32)
  val_dataset = GaitViViTDataset(tensor_df=val_data, frames_per_video=32)
  test_dataset = GaitViViTDataset(tensor_df=test_data, frames_per_video=32)

  train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
  val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
  test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
  dataloaders = {"train": train_dataloader, "val": val_dataloader, "test": test_dataloader}

  # Create an instance of the GaitViViT model and load the VivitForVideoClassification weights.
  model = GaitViViT().to(device)
  hf_model = VivitForVideoClassification.from_pretrained("google/vivit-b-16x2-kinetics400")
  hf_state_dict = hf_model.state_dict()
  translated_state_dict = weight_map(hf_state_dict)
  model.load_state_dict(translated_state_dict, strict=False)

  # Perform linear probing.
  print("--- Phase 1 - Linear Probing ---")
  model = linear_probing(model,
                         dataloaders,
                         loss_function,
                         num_epochs=5,
                         model_path=f"/content/drive/MyDrive/bachelor_thesis/models/best_head_fold_{k + 1}.pth",
                         metrics_path=f"/content/drive/MyDrive/bachelor_thesis/metrics/head_lp_fold_{k + 1}.csv",
                         device=device,
                         checkpoint_path=f"/content/drive/MyDrive/bachelor_thesis/models/lp_resume_checkpoint_fold_{k + 1}.pth")

  # Perform fine-tuning.
  print("--- Phase 2 - Fine-tuning ---")
  model = model_finetuning(model,
                           dataloaders,
                           loss_function,
                           num_epochs=15,
                           head_path=f"/content/drive/MyDrive/bachelor_thesis/models/best_head_fold_{k + 1}.pth",
                           model_path=f"/content/drive/MyDrive/bachelor_thesis/models/best_model_fold_{k + 1}.pth",
                           metrics_path=f"/content/drive/MyDrive/bachelor_thesis/metrics/model_ft_fold_{k + 1}.csv",
                           device=device,
                           checkpoint_path=f"/content/drive/MyDrive/bachelor_thesis/models/ft_resume_checkpoint_fold_{k + 1}.pth")

  # Perform testing.
  print("--- Phase 3 - Testing ---")
  test_results, y_true, y_probs = model_test(model,
                                             dataloaders,
                                             loss_function,
                                             best_model_path=f"/content/drive/MyDrive/bachelor_thesis/models/best_model_fold_{k + 1}.pth",
                                             metrics_path=f"/content/drive/MyDrive/bachelor_thesis/metrics/model_test_fold_{k + 1}.csv",
                                             matrix_path=f"/content/drive/MyDrive/bachelor_thesis/metrics/confusion_matrix_fold_{k + 1}.png",
                                             device=device)

  # Perform the ablation study.
  print("--- Phase 4 - Ablation Study ---")
  ablation_df = ablation_study(y_true,
                               y_probs,
                               thresholds=np.arange(0.3, 0.8, 0.1),
                               csv_path=f"/content/drive/MyDrive/bachelor_thesis/metrics/ablation_study_fold_{k + 1}.csv",
                               plot_path=f"/content/drive/MyDrive/bachelor_thesis/metrics/ablation_study_fold_{k + 1}.png")

  print(f"--- Fold {k + 1} successfully completed. ---")
  torch.cuda.empty_cache()

print("Training successfully completed.")

## 5 - Performance Analysis

After training each version of the model and re-organizing the files, the final step consists of understanding how well the model performs on the task, focusing on the **F1-Score** metric due to the class imbalance of the dataset.

### Testing Metrics

Analysing the testing metrics, as well as their distribution across folds, allows to understand how much each version of the model is able to generalize on unseen data.

In [ ]:
def test_results(i=0, few_shot=False):
  if few_shot:
    # Load the metrics associated to the few-shot model.
    files = [pd.read_csv(f"/content/drive/MyDrive/bachelor_thesis/metrics/few_shot/model_test_1_fold_{k + 1}.csv") for k in range(5)]
  else:
    # Load the metrics associated to one version of the transformer model.
    files = [pd.read_csv(f"/content/drive/MyDrive/bachelor_thesis/metrics/v{i + 1}/model_test_{i + 1}_fold_{k + 1}.csv") for k in range(5)]
  all_data = pd.concat(files, ignore_index=True)

  # Compute mean and standard deviation for precision, recall and F1.
  precision_mean, precision_std = all_data["precision"].mean(), all_data["precision"].std()
  recall_mean, recall_std = all_data["recall"].mean(), all_data["recall"].std()
  f1_mean, f1_std = all_data["f1"].mean(), all_data["f1"].std()

  return {"precision": (precision_mean, precision_std),
          "recall": (recall_mean, recall_std),
          "f1": (f1_mean, f1_std)}

In [ ]:
# Run the function once on the few-shot model.
print("--- Few-Shot Model ---")
values = test_results(i=0, few_shot=True)
print(f"Precision: {values['precision']}")
print(f"Recall: {values['recall']}")
print(f"F1: {values['f1']}")

# Iterate through each version of the transformer model.
print("--- Transformer Model ---")
for i in range(5):
  print(f"--- Version {i + 1} ---")
  values = test_results(i=i, few_shot=False)
  print(f"Precision: {values['precision']}")
  print(f"Recall: {values['recall']}")
  print(f"F1: {values['f1']}")

### Training and Validation Performance

Measuring how training and validation performance varies throughout the fine-tuning phase can come in handy to spot eventual patterns in early stopping triggers or hyperparameter combinations.

In [ ]:
def metric_plot(mode, metric="f1", i=0, few_shot=False, plot_path="/content/drive/MyDrive/bachelor_thesis/results/performance.png"):
  if few_shot:
    # Load the metrics associated to the few-shot model.
    files = [pd.read_csv(f"/content/drive/MyDrive/bachelor_thesis/metrics/few_shot/model_ft_1_fold_{k + 1}.csv") for k in range(5)]
  else:
    # Load the metrics associated to one version of the transformer model.
    files = [pd.read_csv(f"/content/drive/MyDrive/bachelor_thesis/metrics/v{i + 1}/model_ft_{i + 1}_fold_{k + 1}.csv") for k in range(5)]
  all_data = pd.concat(files, ignore_index=True)

  # Create a new column to identify hyperparameter combinations.
  all_data["params"] = all_data.apply(lambda r: f"lr={r["start_lr"]}, wd={r["weight_decay"]}", axis=1)

  # Create the line plot, using the errorbar="sd" argument to plot mean and standard deviation.
  plt.figure(figsize=(10, 6))
  sns.lineplot(
      data=all_data,
      x="epoch",
      y=metric,
      hue="params",
      style="params",
      errorbar="sd")
  plt.title(f"{mode.capitalize} F1-Score Across Epochs")
  plt.xlabel("Epoch")
  plt.ylabel("F1-Score")
  plt.grid(True, linestyle="--", alpha=0.6)
  plt.legend(title="Hyperparameters", bbox_to_anchor=(1.05, 1), loc="upper left")
  plt.tight_layout()

  # Save the line plot.
  os.makedirs(plot_path, exist_ok=True)
  plt.savefig(plot_path, dpi=300)
  plt.show()

def training_plot(i=0, few_shot=False, plot_path="/content/drive/MyDrive/bachelor_thesis/results/performance.png"):
  # Create a line plot for training F1.
  metric_plot("training", "train_f1", i, few_shot, plot_path)

def validation_plot(i=0, few_shot=False, plot_path="/content/drive/MyDrive/bachelor_thesis/results/performance.png"):
  # Create a line plot for validation F1.
  metric_plot("validation", "val_f1", i, few_shot, plot_path)

In [ ]:
# Run the functions once on the few-shot model.
print("--- Few-Shot Model ---")
training_plot(i=0, few_shot=True, plot_path="/content/drive/MyDrive/bachelor_thesis/results/few_shot/training_performance_few_shot.png")
validation_plot(i=0, few_shot=True, plot_path="/content/drive/MyDrive/bachelor_thesis/results/few_shot/validation_performance_few_shot.png")

# Iterate through each version of the transformer model.
print("--- Transformer Model ---")
for i in range(5):
  print(f"--- Version {i + 1} ---")
  training_plot(i=i, few_shot=False, plot_path=f"/content/drive/MyDrive/bachelor_thesis/results/v{i + 1}/training_performance_{i + 1}.png")
  validation_plot(i=i, few_shot=False, plot_path=f"/content/drive/MyDrive/bachelor_thesis/results/v{i + 1}/validation_performance_{i + 1}.png")

### Ablation Study

Since this project aims to serve a medical purpose, an ablation study is conducted in order to determine which classification threshold can be the most suitable for this model, based on two general ideas:
1. **Lowering the classification threshold** makes the model more sensitive to positive examples, which can come in handy for early detection of Parkinson's disease, although the model also becomes more vulnerable to false positives.
2. **Increasing the classification threshold** requires more confidence towards positive examples, reducing the risk of false positives, although the model may not be able to detect early or mild symptoms of Parkinson's disease.

In [ ]:
def ablation_plot(i=0, few_shot=False, plot_path="/content/drive/MyDrive/bachelor_thesis/results/ablation_study.png"):
  if few_shot:
    # Load the metrics associated to the few-shot model.
    files = [pd.read_csv(f"/content/drive/MyDrive/bachelor_thesis/metrics/few_shot/ablation_study_1_fold_{k + 1}.csv") for k in range(5)]
  else:
    # Load the metrics associated to one version of the transformer model.
    files = [pd.read_csv(f"/content/drive/MyDrive/bachelor_thesis/metrics/v{i + 1}/ablation_study_{i + 1}_fold_{k + 1}.csv") for k in range(5)]
  all_data = pd.concat(files, ignore_index=True)

  # Use the melt function to plot precision, recall and F1 together with respect to the identifier threshold.
  melted_data = pd.melt(all_data,
                        id_vars=["threshold"],
                        value_vars=["precision", "recall", "f1"],
                        var_name="Metric",
                        value_name="Score")

  # Create the line plot, using the errorbar="sd" argument to plot mean and standard deviation.
  plt.figure(figsize=(9, 5))
  sns.lineplot(data=melted_data,
               x="threshold",
               y="Score",
               hue="Metric",
               style="Metric",
               errorbar="sd")
  plt.title("Ablation Study - Performance vs Classification Threshold")
  plt.xlabel("Classification Threshold")
  plt.ylabel("Score")
  plt.grid(True, linestyle="--". alpha=0.6)
  plt.tight_layout()

  # Save the line plot.
  os.makedirs(plot_path, exist_ok=True)
  plt.savefig(plot_path, dpi=300)
  plt.show()

In [ ]:
# Run the function once on the few-shot model.
print("--- Few-Shot Model ---")
ablation_plot(i=0, few_shot=True, plot_path="/content/drive/MyDrive/bachelor_thesis/results/few_shot/ablation_study_few_shot.png")

# Iterate through each version of the transformer model.
print("--- Transformer Model ---")
for i in range(5):
  print(f"--- Version {i + 1} ---")
  ablation_plot(i=i, few_shot=False, plot_path=f"/content/drive/MyDrive/bachelor_thesis/results/v{i + 1}/ablation_study_{i + 1}.png")